In [1]:
from agent1.classification.tfidf_classifier import TfidfIntentClassifier
from agent1.classification.semantic_classifier import SemanticIntentClassifier
from agent1.classification.ensemble import IntentClassifierEnsemble

In [2]:
from pathlib import Path
import pandas as pd

# Notebook is in agent1/notebooks/scratch/
# PROJECT_ROOT = Path.cwd().parents[1]

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "Bitext-retail-ecommerce-llm-chatbot-training-dataset.parquet"
)

In [3]:
print(PROJECT_ROOT)
print(DATA_PATH)

d:\Projects\projects\agent1
d:\Projects\projects\agent1\data\raw\Bitext-retail-ecommerce-llm-chatbot-training-dataset.parquet


In [4]:
# D:\Projects\projects\agent1\data\raw\Bitext-retail-ecommerce-llm-chatbot-training-dataset.parquet
df = pd.read_parquet(
    DATA_PATH,
    columns=["instruction", "intent", "category"]
)

In [5]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["intent"]
)

In [6]:
X_train = train_df["instruction"]
X_test  = test_df["instruction"]

y_intent_train = train_df["intent"]
y_intent_test  = test_df["intent"]

In [7]:
import numpy as np
EMBEDDINGS_DIR = PROJECT_ROOT / "data" / "processed" / "embeddings"
#Loading embeddings whenever we need to use them again, without recomputing:
X_train_emb = np.load(
    EMBEDDINGS_DIR / "X_train_embeddings.npy"
)

X_test_emb = np.load(
    EMBEDDINGS_DIR / "X_test_embeddings.npy"
)

In [ ]:
#fit the tfidf model
tfidf_model=TfidfIntentClassifier()
tfidf_model.fit(X_train, y_intent_train)

#fit the semantic model
semantic_model=SemanticIntentClassifier()
semantic_model.fit(X_train_emb, y_intent_train)




In [ ]:
# fit the ensemble model
ensemble_model = IntentClassifierEnsemble(
    tfidf_params={
        "ngram_range": (1, 2),
        "min_df": 2,
        "sublinear_tf": True,
        "max_iter": 1000,
    },
    semantic_params={
        "model_name": "sentence-transformers/all-MiniLM-L6-v2",
        "max_iter": 1000,
        "device": None,
    },
    tfidf_weight=0.5,
)

ensemble_model.fit(X_train, y_intent_train)

In [ ]:
#evaluation
from sklearn.metrics import accuracy_score, classification_report

y_pred_intent = tfidf_model.predict(X_test)

print(
    "tfidf accuracy:",
    accuracy_score(y_intent_test, y_pred_intent)
)

print(
    classification_report(
        y_intent_test,
        y_pred_intent
    )
)

In [ ]:
y_pred_semantic = semantic_model.predict(
    X_test_emb
)

semantic_accuracy = accuracy_score(
    y_intent_test,
    y_pred_semantic
)

print(f"Semantic intent accuracy: {semantic_accuracy:.4f}")

In [ ]:
y_pred_ensemble = ensemble_model.predict(
    X_test
)

ensemble_accuracy = accuracy_score(
    y_intent_test,
    y_pred_ensemble
)

print(f"Ensemble intent accuracy: {ensemble_accuracy:.4f}")

In [ ]:
print(ensemble_model.predict(["Can you help me find a new pair of running shoes?"]))